In [1]:
import pandas as pd

In [4]:
pd.set_option("display.max_rows", None)

In [ ]:
import pandas as pd

# ============================================================
# 1. VERİ YÜKLEME
# ============================================================
with open("/Users/YGT/ist-airport-decision-support-system/data/bronze/weather/allmetartaf.txt", "r") as f:
    lines = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]

df_metar_raw = pd.DataFrame({"raw": lines})
df_metar_raw = df_metar_raw[df_metar_raw["raw"].str[0].str.isdigit()].reset_index(drop=True)

# ============================================================
# 2. PARSE
# ============================================================
df_metar_raw["timestamp"] = pd.to_datetime(
    df_metar_raw["raw"].str[:12],
    format="%Y%m%d%H%M",
    utc=True
)
df_metar_raw["type"] = df_metar_raw["raw"].str.extract(r'\b(METAR|TAF)\b')

df_obs = df_metar_raw[df_metar_raw["type"] == "METAR"].sort_values("timestamp").reset_index(drop=True)
df_taf = df_metar_raw[df_metar_raw["type"] == "TAF"].sort_values("timestamp").reset_index(drop=True)

# ============================================================
# 3. LOOKUP FONKSİYONU
# ============================================================
def get_weather_prompt_at(t, df_obs, df_taf):
    """
    Paper Section 3.2.2:
    - METAR: t'den önceki son 30 dk içinde yayınlanan en güncel gözlem
             bulunamazsa en yakın önceki METAR
    - TAF  : t'den önceki son 6 saat içinde yayınlanan en güncel TAF
    """
    # METAR
    window = df_obs[
        (df_obs["timestamp"] <= t) &
        (df_obs["timestamp"] >= t - pd.Timedelta(minutes=30))
    ]
    if len(window) == 0:
        window = df_obs[df_obs["timestamp"] <= t]

    metar_str = window.iloc[-1]["raw"] if len(window) > 0 else ""

    # TAF
    taf_str = ""
    if len(df_taf) > 0:
        taf_window = df_taf[
            (df_taf["timestamp"] <= t) &
            (df_taf["timestamp"] >= t - pd.Timedelta(hours=6))
        ]
        if len(taf_window) > 0:
            taf_str = taf_window.iloc[-1]["raw"]

    # Paper Figure 10 formatı — ham string, decode etme
    prompt = f"METAR in effect: {metar_str}"
    if taf_str:
        prompt += f" TAF in effect: {taf_str}"

    return prompt


print(f"METAR : {len(df_obs):,} satır → metar_gold.parquet")
print(f"TAF   : {len(df_taf):,} satır → taf_gold.parquet")
print(f"\nKolonlar: {df_obs.columns.tolist()}")
print(f"\nÖrnek METAR:\n{df_obs.iloc[-1]['raw']}")
if len(df_taf) > 0:
    print(f"\nÖrnek TAF:\n{df_taf.iloc[-1]['raw']}")

Kaydedildi.
METAR : 16,319 satır → metar_gold.parquet
TAF   : 1,422 satır → taf_gold.parquet

Kolonlar: ['raw', 'timestamp', 'type']

Örnek METAR:
202601312350 METAR LTFM 312350Z 04011KT 9999 BKN011 BKN025 05/03 Q1012 RESHRA NOSIG=

Örnek TAF:
202601312240 TAF LTFM 312240Z 0100/0206 03012KT 9999 -SHRA BKN011 BKN025


In [ ]:
# ============================================================
# METAR + TAF BİRLEŞTİRME
# Her METAR'a o anki en güncel TAF'ı eşleştir
# ============================================================

def get_active_taf(t, df_taf):
    """t anında geçerli en güncel TAF'ı döndür"""
    window = df_taf[
        (df_taf["timestamp"] <= t) &
        (df_taf["timestamp"] >= t - pd.Timedelta(hours=6))
    ]
    if len(window) == 0:
        # 6 saat bulamazsan en yakın önceki
        window = df_taf[df_taf["timestamp"] <= t]
    return window.iloc[-1]["raw"] if len(window) > 0 else ""

# Her METAR satırına TAF eşleştir
df_obs["taf_raw"] = df_obs["timestamp"].apply(lambda t: get_active_taf(t, df_taf))

# Paper Figure 10 formatında birleşik prompt
df_obs["weather_prompt"] = df_obs.apply(
    lambda row: f"METAR in effect: {row['raw']}" +
                (f" TAF in effect: {row['taf_raw']}" if row["taf_raw"] else ""),
    axis=1
)

# Kontrol
print(df_obs[["timestamp", "weather_prompt"]].tail(3).to_string())

print(f"Kolonlar: {df_obs.columns.tolist()}")

                      timestamp                                                                                                                                                                                 weather_prompt
16316 2026-01-31 22:50:00+00:00  METAR in effect: 202601312250 METAR LTFM 312250Z 05010KT 9999 BKN011 BKN025 05/03 Q1011 RESHRA NOSIG= TAF in effect: 202601312240 TAF LTFM 312240Z 0100/0206 03012KT 9999 -SHRA BKN011 BKN025
16317 2026-01-31 23:20:00+00:00   METAR in effect: 202601312320 METAR LTFM 312320Z 06012KT 9999 -SHRA BKN011 BKN025 05/03 Q1012 NOSIG= TAF in effect: 202601312240 TAF LTFM 312240Z 0100/0206 03012KT 9999 -SHRA BKN011 BKN025
16318 2026-01-31 23:50:00+00:00  METAR in effect: 202601312350 METAR LTFM 312350Z 04011KT 9999 BKN011 BKN025 05/03 Q1012 RESHRA NOSIG= TAF in effect: 202601312240 TAF LTFM 312240Z 0100/0206 03012KT 9999 -SHRA BKN011 BKN025

Güncellendi: metar_gold.parquet
Kolonlar: ['raw', 'timestamp', 'type', 'taf_raw', 'weather_prompt']


In [16]:
df_obs["weather_prompt"][:5]

0    METAR in effect: 202503010020 METAR LTFM 01002...
1    METAR in effect: 202503010050 METAR LTFM 01005...
2    METAR in effect: 202503010120 METAR LTFM 01012...
3    METAR in effect: 202503010150 METAR LTFM 01015...
4    METAR in effect: 202503010220 METAR LTFM 01022...
Name: weather_prompt, dtype: str